# Лабораторная работа 2. Исследовательский анализ данных

**Вариант 14.** Набор данных `clients.csv` содержит сведения о клиентах магазина: идентификатор, год рождения, образование, семейное положение, доход семьи, количество детей, дату регистрации и количество покупок по акциям.

Цель работы — исследовать распределения и взаимосвязи признаков средствами pandas, matplotlib и seaborn.


## 1 Загрузка и предварительная обработка


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('clients.csv', sep=';')
df.info()
df.describe(include='all')


In [ ]:
df.columns = df.columns.str.lower()
df = df.drop_duplicates().reset_index(drop=True)
df['education'] = df['education'].str.strip().replace({'2n Cycle': 'Graduation'})
df['marital_status'] = df['marital_status'].str.strip().replace({'MARRIED':'Married', 'SINGL':'Single', 'Alone':'Single'})
df = df.dropna(subset=['dt_customer']).reset_index(drop=True)
for col in ['income', 'kidhome', 'numdealspurchases']:
    df[col] = df[col].fillna(df[col].median())
df['dt_customer'] = pd.to_datetime(df['dt_customer'], format='%d.%m.%Y')
df['income'] = df['income'].round().astype(int)
df['kidhome'] = df['kidhome'].round().astype(int)
df['numdealspurchases'] = df['numdealspurchases'].round().astype(int)
df['income_level'] = pd.qcut(df['income'], q=3, labels=['Низкий', 'Средний', 'Высокий'])
df.head()


## 2 Матрица рассеяния и категориальная диаграмма


In [ ]:
pd.plotting.scatter_matrix(df[['year_birth','income','kidhome','numdealspurchases']], figsize=(10,10), diagonal='hist')
plt.show()
sns.scatterplot(data=df, x='year_birth', y='income', hue='education')
plt.show()


Матрица показывает попарные связи числовых признаков. На диагонали видны одномерные распределения, а вне диагонали — облака наблюдений для каждой пары. Связь income и year_birth слабая (r = -0.143), а обратная связь income и kidhome заметнее (r = -0.528), хотя это не доказывает причинную связь. Между числом детей и покупками по акциям наблюдается слабая положительная связь (r = 0.198). Scatter plot с цветовой кодировкой позволяет сравнить доходы клиентов разных образовательных групп и увидеть разброс внутри каждой группы.


**Развёрнутый вывод по рисункам 1–2.** Матрица рассеяния показывает, что доход распределён значительно шире, чем количество детей и покупок по акциям. Большая часть клиентов имеет 0–1 ребёнка и 0–3 покупки по акциям, тогда как доход принимает значения от 2 447 до 162 397. Облака точек сильно перекрываются, поэтому год рождения и образование сами по себе не объясняют весь разброс доходов. Обратная связь дохода и количества детей является особенностью этой выборки и требует осторожной интерпретации.


## 3 Гистограммы, корреляция и ковариация


In [ ]:
numeric = ['year_birth','income','kidhome','numdealspurchases']
df[numeric].hist(bins=12, figsize=(12,8)); plt.show()
print('Матрица корреляции:')
print(df[numeric].corr().round(3))
print('Матрица ковариации:')
print(df[numeric].cov().round(2))
sns.heatmap(df[numeric].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.show()


**Развёрнутый вывод по рисунку 3 и тепловой карте.** Гистограмма income имеет правый хвост: медиана равна 52 614, среднее — 53 129.80, а отдельные высокие значения немного увеличивают среднее. Для kidhome медиана равна 0, для numdealspurchases — 2, поэтому типичный клиент не имеет детей дома и совершает около двух покупок по акциям. Тепловая карта подтверждает, что большинство линейных связей слабые: кроме income — kidhome, остальные коэффициенты близки к нулю. Ковариация зависит от единиц измерения, поэтому для сравнения использована корреляция.


## 4 Дополнительные виды графиков


In [ ]:
df.plot(x='income', y='numdealspurchases', kind='hexbin', gridsize=18, cmap='viridis', figsize=(9,6), sharex=False)
plt.show()
sns.boxplot(x=df['income']); plt.show()
sns.boxplot(data=df, x='income_level', y='income', hue='income_level', legend=False); plt.show()
sns.boxplot(data=df, x='education', y='income', hue='education', legend=False); plt.show()
df.boxplot(column='income', by='marital_status', figsize=(10,6)); plt.suptitle(''); plt.show()


**Развёрнутый вывод по рисункам 5–9.** Hexbin показывает концентрацию клиентов около дохода 30–70 тыс. и 0–4 покупок по акциям; редкие ячейки справа соответствуют высокодоходным клиентам. Общий boxplot показывает медиану дохода 52 614 и широкий разброс с верхними выбросами. Квантили разделили клиентов на почти равные уровни дохода (264, 263 и 264 наблюдения), но расстояния между уровнями не одинаковы в денежных единицах. У Basic медиана дохода 24 279, у PhD — 57 424.5; при этом диапазоны образовательных групп сильно перекрываются. Средние доходы по семейному положению находятся в диапазоне от 50 990.66 до 55 452.45, поэтому этот признак полезен для описания сегментов, но недостаточен для предсказания дохода.


## 5 Задания варианта 14


In [ ]:
# Задание 1: количество клиентов по образованию и семейному статусу
education_order = sorted(df['education'].unique())
marital_order = sorted(df['marital_status'].unique())
all_combinations = pd.MultiIndex.from_product(
    [education_order, marital_order],
    names=['education', 'marital_status']
)
task1 = (df.groupby(['education', 'marital_status']).size()
         .reindex(all_combinations, fill_value=0)
         .rename('count')
         .reset_index())
print(task1)
plt.figure(figsize=(10, 6))
sns.barplot(data=task1, x='education', y='count', hue='marital_status',
            order=education_order, hue_order=marital_order,
            estimator=sum, errorbar=None)
plt.title('Количество клиентов по образованию и семейному статусу')
plt.xlabel('Уровень образования')
plt.ylabel('Количество клиентов')
plt.show()

# Задание 2: средний доход по семейному положению
task2 = df.pivot_table(index='marital_status', values='income', aggfunc='mean').round(2)
print(task2)
task2.plot(kind='line', linestyle='', marker='*', markersize=20, color='deeppink', figsize=(10,6)); plt.show()

# Задание 3: образование клиентов с детьми
task3 = df[df['kidhome'] > 0]['education'].value_counts()
print(task3)
plt.figure(figsize=(10, 6))
plt.pie(task3.values, labels=task3.index, autopct='%1.1f%%', startangle=90)
plt.title('Уровень образования клиентов, имеющих детей')
plt.ylabel('')
plt.axis('equal')
plt.show()


**Развёрнутый вывод по рисункам 10–12.** На countplot наиболее многочисленна категория Graduation — 439 клиентов (55.5%); крупнейшая отдельная комбинация — Graduation и Married (170 клиентов). Группа Basic мала (15 наблюдений), поэтому её сравнения требуют осторожности. На графике средних доходов максимум у Widow (55 452.45), минимум у Single (50 990.66), однако Widow представлена только 29 клиентами, и среднее чувствительно к отдельным значениям. В третьем задании после фильтрации клиентов с kidhome > 0 круговая диаграмма построена средствами matplotlib (plt.pie); Graduation составляет 183 человека (56.5%), PhD — 21.3%, Master — 19.4%, Basic — 2.8%. Эти доли относятся только к отфильтрованной подгруппе.


### Пояснение к каждому рисунку

1. **Рисунок 1, матрица рассеяния.** Доход имеет самый широкий разброс; наиболее заметна обратная связь income и kidhome (r = -0.528), но она не означает причинность.
2. **Рисунок 2, scatter plot по образованию.** Образовательные группы перекрываются; у PhD медиана дохода выше, у Basic ниже, но внутри групп сохраняется большой разброс.
3. **Рисунок 3, гистограммы.** Доход асимметричен вправо, kidhome и numdealspurchases принимают дискретные значения; медианы равны 0 и 2 соответственно.
4. **Рисунок 4, тепловая карта.** Большинство корреляций близки к нулю; исключение — обратная связь income и kidhome.
5. **Рисунок 5, hexbin.** Основная плотность наблюдений находится около дохода 30–70 тыс. и 0–4 покупок по акциям.
6. **Рисунок 6, общий boxplot дохода.** Медиана равна 52 614, центральная половина доходов лежит примерно между 36 257 и 69 119; справа есть высокие выбросы.
7. **Рисунок 7, boxplot по уровням дохода.** Квантили образуют почти равные группы, но денежные интервалы между уровнями различаются.
8. **Рисунок 8, boxplot по образованию.** Медиана Basic минимальна (24 279), PhD максимальна (57 424.5), однако распределения заметно перекрываются.
9. **Рисунок 9, boxplot по семейному положению.** Средние доходы различаются умеренно — от 50 990.66 у Single до 55 452.45 у Widow.
10. **Рисунок 10, countplot.** Graduation — крупнейшая образовательная группа (439 клиентов), а Graduation и Married — крупнейшая комбинация (170 клиентов).
11. **Рисунок 11, средний доход по семейному положению.** Максимум у Widow, но группа мала (29 клиентов), поэтому результат чувствителен к отдельным значениям.
12. **Рисунок 12, pie chart клиентов с детьми.** Graduation занимает 56.5% подгруппы; доли относятся только к клиентам с kidhome > 0 и не переносятся на всю выборку.


## 6 Вывод


После загрузки и очистки данных были обработаны пропуски, дубликаты, категориальные значения и дата регистрации; в итоговой таблице осталось 791 наблюдение. Распределения дохода, количества детей и покупок по акциям изучены с помощью гистограмм и boxplot, а взаимосвязи — с помощью scatter plot, hexbin и матрицы корреляций. Анализ показал правостороннюю асимметрию дохода, слабые линейные связи большинства признаков и более заметную обратную связь income с kidhome. Образование и семейное положение связаны с различиями дохода описательно, но сильное перекрытие распределений показывает, что нужны дополнительные признаки. Для варианта 14 построены группировка, сводный график и круговая диаграмма; каждый график дополнен интерпретацией и ограничениями вывода.
